In [1]:
# from datasets import load_dataset

# ds = load_dataset("HuggingFaceM4/FairFace", "0.25")

In [2]:
# import os
# os.makedirs("fairface_data", exist_ok=True)
# for i in range(100):
#     img = ds["train"][i]["image"]
#     img.save(f"fairface_data/{i:05d}.jpg")

In [3]:
import os
import numpy as np
from PIL import Image

def hstack(images):
    if len(images) == 0:
        raise ValueError("Need 0 or more images")

    if isinstance(images[0], np.ndarray):
        images = [Image.fromarray(img) for img in images]
    width = sum([img.size[0] for img in images])
    height = max([img.size[1] for img in images])
    stacked = Image.new(images[0].mode, (width, height))

    x_pos = 0
    for img in images:
        stacked.paste(img, (x_pos, 0))
        x_pos += img.size[0]
    return stacked


import sys
sys.path.append("../../third_party/arc2face")
from diffusers import (
    StableDiffusionPipeline,
    UNet2DConditionModel,
    DPMSolverMultistepScheduler,
)

from arc2face import CLIPTextModelWrapper, project_face_embs

import torch
from insightface.app import FaceAnalysis
from PIL import Image
import numpy as np

base_model = 'stable-diffusion-v1-5/stable-diffusion-v1-5'

encoder = CLIPTextModelWrapper.from_pretrained(
    '../../checkpoints/arc2face', subfolder="encoder", torch_dtype=torch.float16
)

unet = UNet2DConditionModel.from_pretrained(
    '../../checkpoints/arc2face', subfolder="arc2face", torch_dtype=torch.float16
)

pipeline = StableDiffusionPipeline.from_pretrained(
        base_model,
        text_encoder=encoder,
        unet=unet,
        torch_dtype=torch.float16,
        safety_checker=None
    )

/home/user/miniconda/envs/arc2face/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading pipeline components...: 100%|██████████| 6/6 [00:00<00:00, 23.93it/s]
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


In [4]:
import requests
from io import BytesIO
import deepface

import cv2
import torch

In [5]:
pipeline.scheduler = DPMSolverMultistepScheduler.from_config(pipeline.scheduler.config)
pipeline = pipeline.to('cuda')

In [13]:
import pickle
with open("./test.pkl", "rb") as f: 
    id_embs = pickle.load(f) 
id_embs["filenames"] = sum(id_embs["filenames"], [])

fn = id_embs["filenames"][0]
# fn = sum(id_embs["filenames"], [])
fn = [i.replace("./", "./") for i in fn] 

id_embs["student_embeddings"] = torch.cat(id_embs["student_embeddings"], dim=0)
templates = id_embs["templates"] 

templates = torch.cat(templates, dim=0) 

In [14]:
import wandb
os.makedirs("attributes_fracface/gen", exist_ok=True)
os.makedirs("attributes_fracface/in", exist_ok=True)
 
for idx in range(100):
    id_emb = id_embs["student_embeddings"][idx][None,:].to(torch.float16)
    id_emb = (id_emb/torch.norm(id_emb, dim=1, keepdim=True)).to(torch.float16).to("cuda")
    id_emb = project_face_embs(pipeline, id_emb)    # pass through the encoder
    print(id_emb.mean())
    num_images = 1  
    image = pipeline(prompt_embeds=id_emb, num_inference_steps=25, guidance_scale=3.0, num_images_per_prompt=1).images[0]
    Image.open(id_embs["filenames"][idx].replace("fracface", ".")).save(f"attributes_fracface/in/{idx:05d}.png")
    image.save(f"attributes_fracface/gen/{idx:05d}.png")

tensor(-0.1094, device='cuda:0', dtype=torch.float16)


100%|██████████| 25/25 [00:03<00:00,  8.12it/s]


tensor(-0.1107, device='cuda:0', dtype=torch.float16)


100%|██████████| 25/25 [00:03<00:00,  8.08it/s]


tensor(-0.1099, device='cuda:0', dtype=torch.float16)


...

100%|██████████| 25/25 [00:03<00:00,  8.10it/s]


tensor(-0.1099, device='cuda:0', dtype=torch.float16)


100%|██████████| 25/25 [00:02<00:00,  8.91it/s]


tensor(-0.1095, device='cuda:0', dtype=torch.float16)


100%|██████████| 25/25 [00:03<00:00,  8.16it/s]


In [ ]:
# switch to neg env
from deepface import DeepFace
import numpy as np
age_mae = []
race_acc = []
gender_acc = []
emotion_acc = []
for i in range(500):
  res1 = DeepFace.analyze(
    img_path = f"attributes/gen/{i:05d}.png", actions = ['age', 'gender', 'race', 'emotion'], detector_backend="mtcnn", enforce_detection=False
  )
  res2 = DeepFace.analyze(
    img_path = f"attributes/in/{i:05d}.png", actions = ['age', 'gender', 'race', 'emotion'], detector_backend="mtcnn", enforce_detection=False
  )
  age_mae.append(abs(res1[0]['age'] - res2[0]['age']))
  race_acc.append(int(res1[0]['dominant_race'] == res2[0]['dominant_race']))
  gender_acc.append(int(res1[0]['dominant_gender'] == res2[0]['dominant_gender']))
  emotion_acc.append(int(res1[0]['dominant_emotion'] == res2[0]['dominant_emotion']))
  print(f"Age MAE: {np.mean(age_mae):.2f}, Race Acc: {np.mean(race_acc)*100:.2f}, Gender Acc: {np.mean(gender_acc)*100:.2f}, Emotion Acc: {np.mean(emotion_acc)*100:.2f}", end="\r")

In [23]:
res2[0]["dominant_race"]

'asian'